In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Row
data = [Row(order_id=101, order_date="2025-12-10", ship_date="2025-12-18", amount=5000),
    Row(order_id=102, order_date="2025-12-18", ship_date="2025-12-25", amount=8000),
    Row(order_id=103, order_date="2025-12-20", ship_date="2025-12-22", amount=3000)]

df = spark.createDataFrame(data)
df.show()

+--------+----------+----------+------+
|order_id|order_date| ship_date|amount|
+--------+----------+----------+------+
|     101|2025-12-10|2025-12-18|  5000|
|     102|2025-12-18|2025-12-25|  8000|
|     103|2025-12-20|2025-12-22|  3000|
+--------+----------+----------+------+



In [0]:
df.withColumn("today_date", current_date())\
    .withColumn("today_timestamp", current_timestamp()).show()

+--------+----------+----------+------+----------+--------------------+
|order_id|order_date| ship_date|amount|today_date|     today_timestamp|
+--------+----------+----------+------+----------+--------------------+
|     101|2025-12-10|2025-12-18|  5000|2026-01-27|2026-01-27 13:47:...|
|     102|2025-12-18|2025-12-25|  8000|2026-01-27|2026-01-27 13:47:...|
|     103|2025-12-20|2025-12-22|  3000|2026-01-27|2026-01-27 13:47:...|
+--------+----------+----------+------+----------+--------------------+



In [0]:
df.printSchema()


root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- amount: long (nullable = true)



In [0]:
df1 = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))\
    .withColumn("ship_date", to_date(col("ship_date"), "yyyy-MM-dd"))
df1.show()

+--------+----------+----------+------+
|order_id|order_date| ship_date|amount|
+--------+----------+----------+------+
|     101|2025-12-10|2025-12-18|  5000|
|     102|2025-12-18|2025-12-25|  8000|
|     103|2025-12-20|2025-12-22|  3000|
+--------+----------+----------+------+



In [0]:
df1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- amount: long (nullable = true)



In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
df1 = df.withColumn("order_date", date_format(col("order_date"), "dd-MM-yyyy"))
df1.show()

+--------+----------+----------+------+
|order_id|order_date| ship_date|amount|
+--------+----------+----------+------+
|     101|10-12-2025|2025-12-18|  5000|
|     102|18-12-2025|2025-12-25|  8000|
|     103|20-12-2025|2025-12-22|  3000|
+--------+----------+----------+------+



In [0]:
df.select("order_id", "order_date",
          year(col("order_date")).alias("order_year"),
          month(col("order_date")).alias("order_month"),
          dayofmonth(col("order_date")).alias("day"),
          weekofyear(col("order_date")).alias("week")).show()

+--------+----------+----------+-----------+---+----+
|order_id|order_date|order_year|order_month|day|week|
+--------+----------+----------+-----------+---+----+
|     101|2025-12-10|      2025|         12| 10|  50|
|     102|2025-12-18|      2025|         12| 18|  51|
|     103|2025-12-20|      2025|         12| 20|  51|
+--------+----------+----------+-----------+---+----+



In [0]:
df.select("order_id", "order_date", date_format("order_date", "EEEE").alias("day_name")\
    ,date_format("order_date", "MMMM").alias("month_name")).show()

+--------+----------+---------+----------+
|order_id|order_date| day_name|month_name|
+--------+----------+---------+----------+
|     101|2025-12-10|Wednesday|  December|
|     102|2025-12-18| Thursday|  December|
|     103|2025-12-20| Saturday|  December|
+--------+----------+---------+----------+



In [0]:
df.select("order_id", "order_date", quarter(col("order_date")).alias("month_quarter")).show()

+--------+----------+-------------+
|order_id|order_date|month_quarter|
+--------+----------+-------------+
|     101|2025-12-10|            4|
|     102|2025-12-18|            4|
|     103|2025-12-20|            4|
+--------+----------+-------------+



In [0]:
df.select("order_id", "order_date", "ship_date", datediff(col("ship_date"),col("order_date")).alias("date_diff")).show()

+--------+----------+----------+---------+
|order_id|order_date| ship_date|date_diff|
+--------+----------+----------+---------+
|     101|2025-12-10|2025-12-18|        8|
|     102|2025-12-18|2025-12-25|        7|
|     103|2025-12-20|2025-12-22|        2|
+--------+----------+----------+---------+



In [0]:
df1 = df.withColumn("new_ship_date", dateadd(col("ship_date"), 120))
df1.show()

+--------+----------+----------+------+-------------+
|order_id|order_date| ship_date|amount|new_ship_date|
+--------+----------+----------+------+-------------+
|     101|2025-12-10|2025-12-18|  5000|   2026-04-17|
|     102|2025-12-18|2025-12-25|  8000|   2026-04-24|
|     103|2025-12-20|2025-12-22|  3000|   2026-04-21|
+--------+----------+----------+------+-------------+



In [0]:
df1.withColumn("months_between",
    months_between(col("new_ship_date"), current_date()).cast("int")).show()

+--------+----------+----------+------+-------------+--------------+
|order_id|order_date| ship_date|amount|new_ship_date|months_between|
+--------+----------+----------+------+-------------+--------------+
|     101|2025-12-10|2025-12-18|  5000|   2026-04-17|             2|
|     102|2025-12-18|2025-12-25|  8000|   2026-04-24|             2|
|     103|2025-12-20|2025-12-22|  3000|   2026-04-21|             2|
+--------+----------+----------+------+-------------+--------------+



In [0]:
df.filter(col("order_date") >= date_sub(current_date(), 30)).show()

+--------+----------+---------+------+
|order_id|order_date|ship_date|amount|
+--------+----------+---------+------+
+--------+----------+---------+------+



In [0]:
df.withColumn("first_day_of_month",trunc(col("order_date"),"MM"))\
    .withColumn("first_day_of_year",trunc(col("order_date"),"YYYY"))\
        .withColumn("last_day_of_month",last_day(col("order_date"))).show()

+--------+----------+----------+------+------------------+-----------------+-----------------+
|order_id|order_date| ship_date|amount|first_day_of_month|first_day_of_year|last_day_of_month|
+--------+----------+----------+------+------------------+-----------------+-----------------+
|     101|2025-12-10|2025-12-18|  5000|        2025-12-01|       2025-01-01|       2025-12-31|
|     102|2025-12-18|2025-12-25|  8000|        2025-12-01|       2025-01-01|       2025-12-31|
|     103|2025-12-20|2025-12-22|  3000|        2025-12-01|       2025-01-01|       2025-12-31|
+--------+----------+----------+------+------------------+-----------------+-----------------+

